# Trabajando con texto (strings)

## 01\. Introducción

En esta lección, aprenderemos un par de otras tareas de limpieza de texto, como:

*   Encontrar cadenas específicas o subcadenas en columnas
*   Extracción de subcadenas de datos no estructurados
*   Eliminar cadenas o subcadenas de una serie

A medida que aprendamos estas tareas, también trabajaremos para generar intuición sobre cómo funcionan estos métodos de cadena para que pueda explorar métodos que no hemos cubierto explícitamente por su cuenta.

Trabajaremos nuevamente con el Informe sobre [la felicidad en el mundo 2015](https://drive.google.com/uc?id=1JexbQAX-5KrQs-lzDXXAYBicYlUQjOo_&export=download) _y_ datos económicos adicionales del Banco Mundial. Puede encontrar el conjunto de datos [aquí](https://drive.google.com/uc?id=1qVVDdOwOjcfjjrJUPX1gcgXEy_vn3sGI&export=download).

*   `ShortName` - Nombre del país
*   `Region` - La región a la que pertenece el país
*   `IncomeGroup`: el grupo de ingresos al que pertenece el país, según el ingreso nacional bruto (GNI) per cápita
*   `CurrencyUnit` - Nombre de la moneda del país
*   `SourceOfMostRecentIncomeAndExpenditureData`: el nombre de la encuesta utilizada para recopilar los datos de ingresos y gastos
*   `SpecialNotes`: contiene notas misceláneas sobre los datos

### Ejercicio

Ya hemos leído `World_Happiness_2015.csv` en un dataframe llamado `happiness2015` y `World_dev.csv` en un dataframe llamado `world_dev`.

In [1]:
from pprint import pprint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
happiness2015 = pd.read_csv('World_Happiness_2015.csv')
world_dev = pd.read_csv('World_dev.csv')

*   Utilice la función `pd.merge()` para combinar `happiness2015` y `world_dev`. Guarde el dataframe resultante en `merged`. Como recordatorio, puede utilizar la siguiente sintaxis para combinar los marcos de datos: `pd.merge(left=df1, right=df2, how='left', left_on='left_df_Column_Name', right_on='right_df_Column_Name')`.
    *   Configure el parámetro `left_on` en la columna `Country` de `happiness2015` y el parámetro `right_on` en la columna `ShortName` de `world_dev`.
*   Utilice el [método `DataFrame.rename()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html) para cambiar el nombre de la columna `SourceOfMostRecentIncomeAndExpenditureData` en `merged` a `IESurvey` (¡porque no queremos seguir escribiendo ese nombre largo!).
    *   Ya hemos guardado la asignación en un diccionario llamado `col_renaming`.
    *   Asegúrese de establecer el parámetro `axis` en 1.

In [3]:
col_ranaming = {
    'SourceOfMostRecentIncomeAndExpenditureData': 'IESurvey',
    }

In [4]:
merged = pd.merge(
    left=happiness2015, 
    right=world_dev,
    how='left',
    left_on='Country',
    right_on='ShortName',
    )

In [5]:
merged.rename(
    columns=col_ranaming, 
    inplace=True,
    )

## 02\. Uso de `apply` para transformar cadenas

*   Primero trabajemos con la columna `CurrencyUnit`.
*   Supongamos que quisiéramos extraer la unidad monetaria sin la nacionalidad principal.
*   Por ejemplo, en lugar de "Danish krone" o "Norwegian krone", solo necesitábamos "corona".

Si quisiéramos completar esta tarea solo para una de las cadenas, podríamos usar el [método `string.split()`](https://docs.python.org/3/library/stdtypes.html) de Python:


In [6]:
words = 'Danish krone'

# ['Danish', 'krone']
listwords = words.split()

# la última palabra.
listwords[-1]

'krone'

### Ejercicio

*   Escriba una función llamada `extract_last_word` con los siguientes criterios:
    *   La función debe aceptar un parámetro llamado `element`.
    *   Usa el método `string.split()` para dividir el objeto en una lista. Primero convierta `element` en una cadena de la siguiente manera: `str(element)`.
    *   Devuelve la última palabra de la lista.
*   Utilice el método `Series.apply()` para aplicar la función a la columna `CurrencyUnit`. Guarde el resultado en `merged['Currency Apply']`.
*   Use el método `Series.head()` para imprimir las primeras cinco filas en `merged['Currency Apply']`.

In [7]:
def extract_last_word(element):
    return str(element).split()[-1]

In [8]:
merged['Currency Apply'] = merged['CurrencyUnit'].apply(extract_last_word)
print(merged['Currency Apply'].head())

0     franc
1     krona
2     krone
3     krone
4    dollar
Name: Currency Apply, dtype: object


## 03\. Descripción general de los métodos de cadenas vectorizadas

En el último ejercicio, extrajimos la última palabra de cada elemento en la columna `CurrencyUnit` usando el método `Series.apply()`. Sin embargo, también aprendimos en la última lección que debemos usar métodos vectorizados incorporados (si existen) en lugar del método `Series.apply()` por razones de rendimiento.

En cambio, podríamos haber dividido cada elemento en la columna `CurrencyUnit` en una lista de cadenas con el método [`Series.str.split()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.split.html), el equivalente vectorizado del método `string.split()` de Python:


![](https://res.cloudinary.com/rafaeltorrese/image/upload/v1744392161/data-science-lecture/03-data-cleaning/01-data-cleaning-and-analysis/figs04/Split.png)

De hecho, pandas ha incorporado una serie de métodos vectorizados que realizan las mismas operaciones para cadenas en serie que los métodos de cadenas de Python.

A continuación se muestran algunos métodos de cadena vectorizados comunes, pero puede encontrar la lista completa [aquí](https://pandas.pydata.org/pandas-docs/stable/user_guide/text.html):

| Método              | Descripción                                                                     |
|---------------------|---------------------------------------------------------------------------------|
| Serie.str.split()   | Divide cada elemento de la Serie.                                               |
| Serie.str.strip()   | Elimina los espacios en blanco de cada cadena en la Serie.                      |
| Serie.str.lower()   | Convierte las cadenas de la Serie a minúsculas.                                 |
| Serie.str.upper()   | Convierte las cadenas de la Serie a mayúsculas.                                 |
| Serie.str.get()     | Recupera el i-ésimo elemento de cada elemento de la Serie.                      |
| Serie.str.replace() | Reemplaza una expresión regular o una cadena en la Serie con otra cadena.       |
| Serie.str.cat()     | Concatena cadenas en una Serie.                                                 |
| Serie.str.extract() | Extrae subcadenas de la serie que coinciden con un patrón de expresión regular. |

![](https://res.cloudinary.com/rafaeltorrese/image/upload/v1744392161/data-science-lecture/03-data-cleaning/01-data-cleaning-and-analysis/figs04/Syntax.png)

El atributo `str` indica que cada objeto de la serie debe tratarse como una cadena, sin que tengamos que cambiar explícitamente el tipo a una cadena como hicimos cuando usamos el método `apply`.

Tenga en cuenta que también podemos dividir cada elemento de la serie para extraer caracteres, pero aún necesitaríamos usar el atributo `str`. Por ejemplo, a continuación accedemos a los primeros cinco caracteres de cada elemento de la columna `CurrencyUnit`:

In [9]:
merged['CurrencyUnit'].str[0:5]

0      Swiss
1      Icela
2      Danis
3      Norwe
4      Canad
       ...  
153    Rwand
154    West 
155      NaN
156    Burun
157    West 
Name: CurrencyUnit, Length: 158, dtype: object

También es bueno saber que los métodos de cadena vectorizados se pueden _encadenar_. Por ejemplo, supongamos que necesitamos dividir cada elemento en la columna `CurrencyUnit` en una lista de cadenas usando el método `Series.str.split()` _y_ capitalizar las letras usando el método `Series.str.upper()`. Puede usar la siguiente sintaxis para aplicar más de un método a la vez:

In [10]:
merged['CurrencyUnit'].str.upper().str.split()

0                   [SWISS, FRANC]
1                 [ICELAND, KRONA]
2                  [DANISH, KRONE]
3               [NORWEGIAN, KRONE]
4               [CANADIAN, DOLLAR]
                  ...             
153               [RWANDAN, FRANC]
154    [WEST, AFRICAN, CFA, FRANC]
155                            NaN
156               [BURUNDI, FRANC]
157    [WEST, AFRICAN, CFA, FRANC]
Name: CurrencyUnit, Length: 158, dtype: object

Sin embargo, no olvide incluir `str` antes de cada nombre de método, ¡o obtendrá un error!

### Ejercicio

*   Use el método `Series.str.split()` para dividir la columna `CurrencyUnit` en una lista de palabras y luego use el método `Series.str.get()` para seleccionar solo la última palabra. Asigne el resultado a `merged['Currency Vectorized']`.
*   Usa el método `Series.head()` para imprimir las primeras cinco filas en `merged['Currency Vectorized']`.

In [13]:
merged['Currency Vectorized'] = merged['CurrencyUnit'].str.split().str.get(-1)
print(merged['Currency Vectorized'].head())

0     franc
1     krona
2     krone
3     krone
4    dollar
Name: Currency Vectorized, dtype: object
